In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)
from pyspark.sql import functions as F

In [0]:
def shapes(table, cols, n=6):
    df = spark.table(f"dld.raw.{table}")
    for c in cols:
        shape = F.regexp_replace(F.regexp_replace(F.col(c), "[0-9]", "9"), "[A-Za-z]", "a")
        rows = (df.select(F.coalesce(shape, F.lit("<NULL>")).alias("shape"))
                  .groupBy("shape").count().orderBy(F.desc("count")).limit(n).collect())
        print(f"{table}.{c}: " + " | ".join(f"{r['shape']} ({r['count']})" for r in rows))
    print()

bulk_cols = ["transaction_id", "instance_date", "actual_worth", "procedure_area",
             "meter_sale_price", "rent_value", "has_parking", "no_of_parties_role_1"]
shapes("e1_transactions", bulk_cols)
shapes("e2_transactions", bulk_cols)
shapes("transactions_daily", ["transaction_number", "instance_date", "trans_value", "procedure_area",
                              "actual_area", "parking", "total_buyer", "total_seller"])

In [0]:
pairs = [("trans_group_en", "group_en"), ("procedure_name_en", "procedure_en"),
         ("reg_type_en", "is_offplan_en"), ("property_usage_en", "usage_en"),
         ("property_type_en", "prop_type_en"), ("property_sub_type_en", "prop_sb_type_en"),
         ("rooms_en", "rooms_en"), ("has_parking", "parking")]
for b, d in pairs:
    bulk = {r[0] for r in spark.sql(f"SELECT DISTINCT {b} FROM dld.raw.e2_transactions").collect()}
    daily = {r[0] for r in spark.sql(f"SELECT DISTINCT {d} FROM dld.raw.transactions_daily").collect()}
    print(f"{b}  vs  {d}")
    print(f"  bulk  ({len(bulk)}): {sorted(map(str, bulk))[:15]}")
    print(f"  daily ({len(daily)}): {sorted(map(str, daily))}")
    print(f"  daily words NOT in bulk: {sorted(map(str, daily - bulk))}")
    print()

In [0]:
display(spark.sql("""
  WITH a AS (SELECT DISTINCT transaction_id FROM dld.raw.e1_transactions),
       b AS (SELECT DISTINCT transaction_id FROM dld.raw.e2_transactions)
  SELECT (SELECT count(*) FROM a) AS e1_ids,
         (SELECT count(*) FROM b) AS e2_ids,
         (SELECT count(*) FROM a JOIN b USING (transaction_id)) AS in_both
"""))

In [0]:
dbutils.library.restartPython()

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0

from etl.migrate import run_migrations
run_migrations(spark, f"{repo}/sql/ddl", upto="200")

from etl.conform import conform_all
conform_all(spark)

In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

txt = open(f"{repo}/etl/migrate.py").read()
print("new migrate.py:", "def selected" in txt and "upto" in txt)
print("new test:", "test_upto" in open(f"{repo}/tests/test_migrate.py").read())

In [0]:
display(spark.sql("""
  SELECT source_table, reason, count(*) AS rows
  FROM (SELECT source_table, explode(reasons) AS reason FROM dld.stg.transaction_quarantine)
  GROUP BY source_table, reason ORDER BY rows DESC"""))
display(spark.table("dld.stg.transaction_quarantine").limit(10))

In [0]:
display(spark.sql("""SELECT term, source_term, std_term, note FROM dld.stg.term_map
                     WHERE source_shape = 'daily' ORDER BY std_term IS NOT NULL, term, source_term"""))

In [0]:
cols = ["instance_date", "amount", "procedure_area", "trans_group", "procedure_name",
        "reg_type", "property_type", "property_usage", "has_parking"]
exprs = ", ".join(f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols)
display(spark.sql(f"SELECT source_shape, count(*) AS rows, {exprs} FROM dld.stg.transaction GROUP BY source_shape"))
display(spark.sql("SELECT source_shape, area_source, count(*) AS rows FROM dld.stg.transaction GROUP BY 1, 2 ORDER BY 1, 3 DESC"))

In [0]:
display(spark.sql("""
  SELECT source_table, reason, count(*) AS rows
  FROM (SELECT source_table, explode(reasons) AS reason FROM dld.stg.transaction_quarantine)
  GROUP BY 1, 2 ORDER BY 3 DESC"""))
display(spark.sql("""
  SELECT instance_date, actual_worth, procedure_area, meter_sale_price, no_of_parties_role_1
  FROM dld.raw.e1_transactions LIMIT 5"""))

In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

import importlib, etl.conform
importlib.reload(etl.conform)            # make sure the fixed conform.py is used
from etl.conform import conform_all
conform_all(spark)                       # prints: raw ... = staged ... + quarantined ... -> OK

display(spark.sql("""
  SELECT source_table, reason, count(*) AS rows
  FROM (SELECT source_table, explode(reasons) AS reason FROM dld.stg.transaction_quarantine)
  GROUP BY 1, 2 ORDER BY 3 DESC"""))

display(spark.sql("""SELECT term, source_term, std_term, note FROM dld.stg.term_map
                     WHERE source_shape = 'daily' ORDER BY std_term IS NOT NULL, term, source_term"""))

cols = ["instance_date", "amount", "procedure_area", "trans_group", "procedure_name",
        "reg_type", "property_type", "property_usage", "has_parking"]
exprs = ", ".join(f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols)
display(spark.sql(f"SELECT source_shape, count(*) AS rows, {exprs} FROM dld.stg.transaction GROUP BY source_shape"))

In [0]:
display(spark.sql("""SELECT term, source_term, std_term, note FROM dld.stg.term_map
                     WHERE source_shape = 'daily' ORDER BY std_term IS NOT NULL, term, source_term"""))

cols = ["instance_date", "amount", "procedure_area", "trans_group", "procedure_name",
        "reg_type", "property_type", "property_usage", "has_parking"]
exprs = ", ".join(f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols)
display(spark.sql(f"SELECT source_shape, count(*) AS rows, {exprs} FROM dld.stg.transaction GROUP BY source_shape"))

In [0]:
import sys, os, importlib
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

import etl.conform
importlib.reload(etl.conform)
print("Mortgage override in file:", ("trans_group", "Mortgage") in etl.conform.OVERRIDES)
print("Sale override in file:    ", ("procedure_name", "Sale") in etl.conform.OVERRIDES)

etl.conform.conform_all(spark)